# Roman Integration and Test Archive (RITA) Data Retrieval

------------------

## Learning Goals

By the end of this tutorial, you will:

- Understand how to programmatically access (count, list, or download) Roman Integration and Test Archive (RITA) data from the Calibration and Supplemental Search Interface (CaSSI), by a number of criteria.


## Table of Contents

* [Introduction](#Introduction)
* [Imports](#Imports)
* [Helper Scripts](#Helper-Scripts)
* [Querying CaSSI](#Querying-CaSSI)
    * [Querying with exposure start times](#Querying-with-exposure-start-times)
    * [Querying with test phase(s)](#Querying-with-test-phase(s))
    * [Querying by OTP name (test script)](Querying-by-OTP-name-(test-script))
    * [Querying by fileSetName(s)](#Querying-by-fileSetName(s))
    * [Querying by filename(s)](#Querying-by-filename(s))
* [Downloading Data](#Downloading-Data)
* [Example companion script usage](#Example-companion-script-usage)
* [Additional resources](#Additional-Resources)


## Introduction

This tutorial demonstrates how to retrieve Roman Integration and Test Archive (RITA) data and use it in the context of a Python session.

The [RITA](https://outerspace.stsci.edu/spaces/RAPD/pages/301172617/Obtain+Data+from+RITA) tutorial in the Roman Pre-Launch documentation (restricted access pre-launch) describes how to access these integration and test data available from MAST through the Calibration and Supplemental Search Interface (CaSSI). These data may be searched by a number of criteria. Here, we focus on provenance, OTP name (test script), test phase, detector, and earliest and latest exposure start time.  (Other constraints may be passed directly as a dictionary).


The integration and test data available from CaSSI covers a variety of data products taken during various ground tests, typically specified by "test_script" (also known as the OTP name/number).  This tutorial only focuses on demonstrating how to *query* CaSSI to obtain file counts or lists and how to *download data* from CaSSI; demonstrations of how to work with the retrieved data are beyond ths scope of this tutorial.

Note that this folder includes a companion script; this offers a compact, customizable way to query, count, or download RITA data from the command line or from python scripts/notebooks. Examples using the various options in this script are given at the end of this notebook.

<div class="alert alert-info" style="color:black; border-color:teal;">
Please note that pre-launch, <b>the MAST Roman CaSSI Search API requires authorization to search and download Roman data products.</b> Before we get started, please ensure that:
    
- ***you are authorized to search and download Roman engineering data from MAST.*** If you are not authorized but you think you should be, email the helpdesk at archive@stsci.edu
- ***you have a [MAST token](https://auth.mast.stsci.edu/token) set to the environment variable*** **`MAST_API_TOKEN`**, OR saved in a file (in the current directory) called `mast_api_token.txt`. <b style="color:red">Note:</b> Ensure your token is valid, as these [expire periodically](https://auth.mast.stsci.edu/info).
</div>

    
<div class="alert alert-warning" style="color:black; background-color:#ffc5c5; border-color:red;">
<b>Note:</b> At this time, Roman data are not accessible from the cloud. Downloads will come from MAST servers, and <b>may be large</b>. Download with caution.
</div>


## Imports

This notebook uses the following packages to retreive data: 

* `os` to get MAST API token envirnoment variable
* `cassi_rita_query_download_script`, the companion script to this notebook, to handle query & download processing.

In [ ]:
import os
import cassi_rita_query_download_script as cq

# MAST authorization token:
mast_token = os.getenv("MAST_API_TOKEN")
if mast_token is None:
    try:
        with open("mast_api_token.txt") as f:
            mast_token = f.read().strip()
    except FileNotFoundError:
        raise ValueError("MAST token not found!")

In [ ]:
from importlib import reload

## Helper Scripts


For this tutorial, functions to connect to the CaSSI web service and query/retreive data files are provided only in the companion script (in the interest of space).

These functions, available by importing the script as `cq` above, will be used later in this tutorial. 

The two key functions are `cq.query_cassi_rita()` and `cq.download_cassi_files()`.  The call signatures are printed below.

In [ ]:
help(cq.query_cassi_rita)

In [ ]:
help(cq.download_cassi_files)

## Querying CaSSI

To begin, we will query CaSSI to find files matching specific search criteria.

Available search criteria include OTP name (test script), test phase, detector, and earliest and latest exposure start time.

One required criteria is the **provenance**, which must be one of `TVAC1`, `TVAC2`, or `FPS`.

### Querying with exposure start times

Using the helper function `cq.query_cassi_rita()`, we will start by searching for all test data where the exposure *started* during a specified time frame.

The earliest and latest exposure start times are in UTC, and must be specified in `YYYY-MM-DD[THH:DD:SS]` format, with **T** as a literal cheracter.

Here, we will select earliest and latest exposure start times of 00:03:00 on 2023 Oct 18 to 03:59:59 on 2023 Oct 18, from TVAC1.

This query is constructed and executed as follows:

*(Omitting the time signature in this script defaults to `00:00:00` for the earliest and `23:59:59` for the latest times.)*

In [ ]:
earliest_exposure_start_time = "2023-10-18T03:00:00"
latest_exposure_start_time =   "2023-10-18T03:29:59"

results = cq.query_cassi_rita(
    earliest_exposure_start_time=earliest_exposure_start_time,
    latest_exposure_start_time=latest_exposure_start_time,
    limit=500,           # limit to number of results
    provenance='TVAC1',
    token=mast_token
)

In [ ]:
results

This query returns metadata for 288 files.

### Querying with test phase(s)

Now, let's run a query that instead restricts to the 'COLD_QUAL' test phase of TVAC1.

Test phases must be specified exactly, with multiple types separated by commas and the entire list wrapped in quotes.

In [ ]:
test_phase = "COLD_QUAL"

results = cq.query_cassi_rita(
    test_phase=test_phase,
    limit=500,           # limit to number of results
    provenance='TVAC1',
    token=mast_token
)

In [ ]:
results

This returns only the first 500 results of 150984 files; 
additional queries specifying the an offset with the `offset` keyword can be used to obtain more results.

### Querying by OTP name (test script)

We well now run a query that instead restricts by OTP name (the test script). 

Wildcards are permitted for this constraint by the MAST API, as described in [the documentation](https://outerspace.stsci.edu/spaces/MASTDOCS/pages/217350619/Core+Search+Parameters#CoreSearchParameters-param_numerics).
Multiple values, separated by commas, are also permitted.

In [ ]:
otp_name = "OTP00620_Ramp*"

results = cq.query_cassi_rita(
    otp_name=otp_name,
    limit=500,           # limit to number of results
    provenance='TVAC1',
    token=mast_token
)

In [ ]:
results

### Querying by fileSetName(s)

This script also supports querying by fileSetName (here, typically the grouping over all detectors of a given exposure).

Wildcards are also permitted for this constraint by the MAST API, as described in [the documentation](https://outerspace.stsci.edu/spaces/MASTDOCS/pages/217350619/Core+Search+Parameters#CoreSearchParameters-param_numerics), as are multiple comma-separated values.

In [ ]:
fileSetName = "TVAC1_NOMOPS_WFIDAR_202310180122*"

results = cq.query_cassi_rita(
    fileSetName=fileSetName,
    limit=500,           # limit to number of results
    provenance='TVAC1',
    token=mast_token
)

In [ ]:
results

### Querying by filename(s)

Finally, we will query by file name to obtain only the metadata on specific files.

Again, wildcards are permitted for this constraint by the MAST API, as described in [the documentation](https://outerspace.stsci.edu/spaces/MASTDOCS/pages/217350619/Core+Search+Parameters#CoreSearchParameters-param_numerics).
Multiple values, separated by commas, are also permitted.

In [ ]:
filename = "TVAC1_NOMOPS_WFIDAR_20231018012216_WFI01*"

results = cq.query_cassi_rita(
    filename=filename,
    limit=500,           # limit to number of results
    provenance='TVAC1',
    token=mast_token
)

In [ ]:
filename = "TVAC1_NOMOPS_WFIDAR_20231018012216_WFI01*"

results = cq.query_cassi_rita(
    filename=filename,
    limit=500,           # limit to number of results
    provenance='TVAC1',
    token=mast_token
)

In [ ]:
results

## Downloading Data

To download Roman supplementary & telemetry data, we will use the second helper funciton, `download_cassi_files()`. 

Here we will download the file from the last metadata query, restricting to just the filename matching `TVAC1_NOMOPS_WFIDAR_20231018012216_WFI01*`

(Recall the full function call signature is shown above.)

*(Note this file is ~1.8 GB.)* 

This function returns the status (`0` for success, `1` for partial/full failure), and optionally (controlled by the flag `return_info`), information & statistics on the download (`results`), the list of all successfully-downloaded files (`successful_files`), and the list of all failed URLs (`failed_urls`) --- which can be used to quickly retry only the failed files.

In [ ]:
# Get CaSSI RIT endpoint URL:
endpoint_url = cq.get_cassi_download_url("roman", "rita")

status, results, successful_files, failed_urls = cq.download_cassi_files(
    results['url'].tolist(),         # Resource URL for query above
    endpoint_url=endpoint_url,
    folder="cassi-rita-data/",
    token=mast_token,
    parallel=True,       # Option to enable parallel downloading. Default: True
    max_workers=2,       # Max number of workers for parallel downloading
    overwrite=False,     # Option to overwrite existing downloads matching an output file name. 
                         # Default: False, skipping any re downloads
    return_info=True,    # Option to suppress results/success/failure info return; if False, returns only status
    verbose=True,        # Option to print download status info
)

If all file(s) download successfully, status will be 0 (with 1 denoting errors in one or more downloads).

Further informaiton is available from the `results` DataFrame and `successful_files`/`failed_urls` lists.

In [ ]:
print(status)

In [ ]:
print(results)

In [ ]:
print(successful_files)

In [ ]:
print(failed_urls)

In [ ]:
ls -lhtr cassi-rita-data

The download folder shows the file downloaded successfully.

## Example companion script usage

The companion script provides the functionality demonstrated above, as well as some additional features to aid in examining the metadata queries directly from the command line.

These usage patterns, with example output in certain cases, are detailed below (with code rendering, which will not be executed as part of this notebook).

**List all command line options**
```console
$ python cassi_rita_query_download_script.py.py --help
```

*(output omitted)*

**Search for files by earliest/latest exposure start date list metadata for TVAC1**
```console
$ python cassi_rita_query_download_script.py -p TVAC1 -s 2023-10-18T03:00:00 -e 2023-10-18T03:29:00
```

*(output omitted)*

**Modify limit or offset**
```console
$ python cassi_rita_query_download_script.py -p TVAC1 -s 2023-10-18T03:00:00 -e 2023-10-18T03:29:00 -l 200 -o 200
```

*(output omitted)*

**Search for files by test phase and list metadata**
```console
$ python cassi_rita_query_download_script.py -p TVAC1 --test-phase "COLD_QUAL"
```

*(output omitted)*

**Search for files by OTP name / test script**
```console
$ python cassi_rita_query_download_script.py -p TVAC1 --otp-name "OTP00620_Ramp*"
```

*(output omitted)*

(Note `--test-script` and `--otp-name` can be used interchangibly.)

**Count files selected by OTP name**
```console
$ python cassi_rita_query_download_script.py -p TVAC1 --otp-name "OTP00620_Ramp*" --count -l 5000

Retrieved 666 of 666 total results, starting at offset=0
Total files:                    666
OTP00620_Ramp_TV1a_R1.txt:      666
```

**Count files selected by fileSetName**
```console
$ python cassi_rita_query_download_script.py -p TVAC1 --fileSetName "TVAC1_NOMOPS_WFIDAR_20231018012216" --count

Retrieved 14 of 14 total results, starting at offset=0
Total files:                     14
OTP00620_Ramp_TV1a_R1.txt:       14
```


**Download files selected by filename(s)**
```console
$ python cassi_rita_query_download_script.py -p TVAC1 --filename "TVAC1_NOMOPS_WFIDAR_20231018012216_WFI0[1-2]*" --download --folder "cassi-rita-data"
```

*(Optionally add the `--overwrite` flag to overwrite any existing downloaded files. Otherwise, will recover gracefully and only try to download files that failed at downloading before.)*

## Additional Resources
* The [Roman CaSSI interface](https://mast.stsci.edu/cassi/#/roman), with restricted access pre-launch.
* The restricted-access [RITA](https://outerspace.stsci.edu/spaces/RAPD/pages/301172617/Obtain+Data+from+RITA) tutorial in the Roman Pre-Launch documentation (on innerspace).  This information will be made public at a later date.

## About this Notebook

**Author(s):** Sedona Price and Zach Claytor, and script writing by Eric Switzer, loosely adapted from the [JWST EDB retrieval notebook](https://spacetelescope.github.io/mast_notebooks/notebooks/JWST/Engineering_Database_Retreival/EDB_Retrieval.html) by MAST staff (chiefly Dick Shaw, Peter Forshay, and Bernie Shiao, with additional editing by Thomas Dutkiewicz). <br>
**Keyword(s):** Roman

***
<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/style-guides/master/guides/images/stsci-logo.png" alt="Space Telescope Logo" width="200px"/> 

[Return to top of page](#Roman-Supplementary-&-Telemetry-Data-Retrieval)